# Figure 5 — Source Data Export

**Figure 5** examines the temporal dynamics of the evolution process via
time-binned PSTH analysis.  Panel 5A shows the population-average PSTH at
the first (block 0) and last (block 55) evolution blocks.  Panel 5C shows
how the activation trajectory changes when firing rates are measured in
successive 10 ms time windows rather than the canonical 50–200 ms window.

## Data requirements

> **Raw neural recordings are required to run this notebook.**
> The `.mat`/`.pkl` files are not distributed with the repository.
> Set the `MATROOT` environment variable to the directory containing
> `Both_BigGAN_FC6_Evol_Stats.pkl` and `Both_BigGAN_FC6_Evol_Stats_expsep.pkl`
> before launching Jupyter, or assign the path directly to `matroot` in the
> cell below.
>
> ```bash
> export MATROOT=/path/to/Mat_Statistics
> ```

The exported source data files are written to `../source_data/Fig5/`.

In [ ]:
import os, sys, pickle, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from os.path import join
from scipy.stats import sem

sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), ".."))
from neuro_data_analysis.neural_data_lib import (
    load_neural_data,
    extract_all_evol_trajectory_psth,
    pad_psth_traj,
    get_expstr,
    extract_natref_activation_array,
    extract_evol_activation_array,
    extract_all_evol_trajectory_dyna,
)
from neuro_data_analysis.neural_data_utils import get_all_masks

In [ ]:
# Set the path to the raw neural recording data.
# Override by setting the MATROOT environment variable before launching Jupyter.
matroot = os.environ.get("MATROOT", "")

BFEStats_merge, BFEStats = load_neural_data()

# Build experiment-level metadata and standard inclusion masks
from neuro_data_analysis.neural_data_lib import pad_resp_traj
resp_col, meta_df = extract_all_evol_trajectory_dyna(BFEStats)
Amsk, Bmsk, V1msk, V4msk, ITmsk, length_msk, spc_msk, sucsmsk, \
    bsl_unstable_msk, bsl_stable_msk, validmsk = get_all_masks(meta_df)

outdir = "../source_data/Fig5"
os.makedirs(outdir, exist_ok=True)

print(f"Valid experiments: {validmsk.sum()}")

## Figure 5A: Population PSTHs per block

Extract per-block PSTH trajectories, select the first (block index 0) and
last (block index 55) blocks, normalise by the maximum response, and export
population-mean PSTHs for DeePSim and BigGAN threads.

In [ ]:
psth_col, meta_psth = extract_all_evol_trajectory_psth(BFEStats)

for block_idx in [0, 55]:
    block_psths = []  # accumulate (n_valid, n_threads, T)
    for Expi in meta_df.loc[validmsk, "Expi"]:
        if Expi not in psth_col:
            continue
        psth_exp = psth_col[Expi]      # list of per-block (n_threads, T) arrays
        if len(psth_exp) <= block_idx:
            continue
        block_psths.append(psth_exp[block_idx])   # (n_threads, T)

    block_arr = np.stack(block_psths, axis=0)     # (N, n_threads, T)
    # Normalise each experiment to its maximum across the time axis
    max_resp = block_arr.max(axis=(1, 2), keepdims=True).clip(min=1e-8)
    block_norm = block_arr / max_resp
    pop_mean = block_norm.mean(axis=0)             # (n_threads, T)

    for thr_idx, space_name in enumerate(["DeePSim", "BigGAN"]):
        pd.DataFrame(pop_mean[thr_idx][np.newaxis, :]).to_csv(
            join(outdir,
                 f"Figure5A_src_all_exps_maxnorm_mean_psth_per_block{block_idx:02d}_{space_name}_mean.csv"),
            index=False)

# Save experiment inclusion mask
meta_df[validmsk].to_csv(join(outdir, "Figure5_src_exp_masks_and_succ_labels.csv"), index=False)

print("Figure 5A source data saved.")

## Figure 5C: Time-binned trajectories

Re-extract activation trajectories using successive 10 ms non-overlapping
time windows spanning 0–200 ms post-stimulus onset.  For each window, export
population-mean and SEM normalised trajectories for both GAN threads.

In [ ]:
from neuro_data_analysis.neural_data_lib import pad_resp_traj

# 10 ms bins from 0 to 200 ms
time_windows = [(t, t + 10) for t in range(0, 200, 10)]

for (t_start, t_end) in time_windows:
    rsp_wdw = range(t_start, t_end)
    resp_col_wdw, _ = extract_all_evol_trajectory_dyna(BFEStats, rsp_wdw=rsp_wdw)
    resp_extrap_arr, extrap_mask_arr, _ = pad_resp_traj(resp_col_wdw)

    valid_idx = np.where(validmsk.values)[0]
    resp_valid = resp_extrap_arr[valid_idx]     # (N, max_gen, 6)
    mask_valid = extrap_mask_arr[valid_idx]      # (N, max_gen)

    # Normalise per experiment to its own maximum response
    max_resp = np.nanmax(
        np.where(~mask_valid[:, :, np.newaxis], resp_valid[:, :, :2], np.nan),
        axis=1, keepdims=True
    ).clip(min=1e-8)
    resp_norm = resp_valid[:, :, :2] / max_resp
    sem_norm  = resp_valid[:, :, 2:4] / max_resp
    resp_norm[mask_valid] = np.nan
    sem_norm[mask_valid]  = np.nan

    pop_mean = np.nanmean(resp_norm, axis=0)   # (max_gen, 2)
    pop_sem  = np.nanmean(sem_norm,  axis=0)

    tag = f"{t_start}-{t_end}"
    for thr_idx, space_name in enumerate(["DeePSim", "BigGAN"]):
        pd.DataFrame(pop_mean[:, thr_idx]).to_csv(
            join(outdir, f"Figure5C_src_normresp_wdw_{tag}_{space_name}_mean.csv"), index=False)
        pd.DataFrame(pop_sem[:, thr_idx]).to_csv(
            join(outdir, f"Figure5C_src_normresp_wdw_{tag}_{space_name}_sem.csv"), index=False)

print(f"Figure 5C source data saved for {len(time_windows)} time windows to", outdir)